# Phase 3b — MMDetection Route B (native `mmdet` + RTMDet)

Rebuilds the detection stage on native **MMDetection** (RTMDet-s) instead of Ultralytics,
per the supervisor's ask, and integrates the enhancement/defogging transform as the
thesis's contribution.

Full background: `docs/MMDETECTION_ROUTE_B_RUNBOOK.md` (local-only, not pushed — see repo README).

**Run order:** mount Drive -> clone/pull repo -> §Env setup (Cells 1-7, run once per
session, does NOT survive a Colab disconnect) -> Task 2 (YOLO->COCO) -> Task 3 (config
sanity checks) -> Task 4 (train, costs compute) -> Task 6 (with/without eval, costs compute).

**Runtime:** Colab, **T4 GPU**. Set this before running anything: `Runtime > Change runtime type > T4 GPU`.


## 0. Mount Drive and get the repo

Datasets live on Drive, not in git. Code comes from git.

In [12]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import os

REPO_DIR = '/content/computer_vision'
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/Ib-Programmer/computer_vision.git {REPO_DIR}

%cd {REPO_DIR}


remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 10 (delta 7), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 19.20 KiB | 855.00 KiB/s, done.
From https://github.com/Ib-Programmer/computer_vision
   06343c9..7ee082b  main       -> origin/main
Updating 06343c9..7ee082b
Fast-forward
 notebooks/Phase3b_MMDetection.ipynb | 882 ++++++++++++++++++++++++++++++++----
 scripts/yolo_to_coco.py             |  19 +-
 2 files changed, 806 insertions(+), 95 deletions(-)
/content/computer_vision


In [ ]:
import shutil
import subprocess

nvidia_smi = shutil.which('nvidia-smi')
ok = False
if nvidia_smi:
    try:
        r = subprocess.run([nvidia_smi, '-L'], capture_output=True, text=True)
        ok = r.returncode == 0 and bool(r.stdout.strip())
        if ok:
            print(r.stdout)
    except OSError:
        ok = False

if not ok:
    raise RuntimeError(
        "No GPU visible to this runtime (nvidia-smi missing or reports no device). Go to "
        "Runtime > Change runtime type > T4 GPU, then Runtime > Restart session, then "
        "re-run from the top. (Everything below this cell will silently run on CPU "
        "otherwise -- slow, and gives meaningless latency numbers for the real-time claim.)"
    )


## 1. Environment setup (§3 of the runbook) — run once per session

Colab's default runtime (Python 3.12, torch 2.11, CUDA 12.8) is **incompatible** with the
OpenMMLab 2.x stack. `condacolab` gives us conda, then we build a separate **Python 3.10**
conda env (`mm`) with a pinned stack. The kernel itself stays 3.12 — every mmdet call below
routes through `conda run -n mm`.

Do not deviate from the pinned versions (torch 2.1.0+cu118 / mmcv 2.1.0 / mmdet 3.3.0 /
numpy<2) — see runbook §3 for why each pin exists.


In [ ]:
# CELL 1 — run ALONE. The kernel auto-restarts after this (expected). Do not re-run.
!pip install -q condacolab
import condacolab
condacolab.install()


In [ ]:
# CELL 2 — after the restart: create the 3.10 env
!mamba create -n mm python=3.10 -y || conda create -n mm python=3.10 -y


In [ ]:
# CELL 3 — GATE: must print 3.10.x before continuing. Stop here if it doesn't.
!conda run -n mm python -c "import sys; print('env Python:', sys.version.split()[0])"


In [ ]:
# CELL 4 — pinned torch (cu118 runs fine under the T4's 12.8 driver)
!conda run -n mm pip install -q torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 \
    --index-url https://download.pytorch.org/whl/cu118
!conda run -n mm pip install -q "numpy<2"
!conda run -n mm python -c "import numpy,torch; print('numpy',numpy.__version__,'| torch',torch.__version__,'| CUDA',torch.cuda.is_available())"


In [ ]:
# CELL 5 — OpenMMLab stack (mmcv from the matching prebuilt index)
!conda run -n mm pip install -q -U openmim
!conda run -n mm mim install mmengine
!conda run -n mm mim install "mmcv==2.1.0" -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.1/index.html
!conda run -n mm mim install "mmdet==3.3.0"


In [ ]:
# CELL 6 — RE-PIN numpy: installing the stack drags numpy back to 2.x, which breaks it.
!conda run -n mm pip install -q "numpy<2"


In [ ]:
# CELL 7 — smoke test. Success = "OK -- native MMDetection works."
!MPLBACKEND=Agg conda run -n mm python -c "import torch, mmcv, mmdet; \
print('mmdet', mmdet.__version__, '| CUDA', torch.cuda.is_available()); \
from mmdet.apis import DetInferencer; DetInferencer('rtmdet_tiny_8xb32-300e_coco'); \
print('OK -- native MMDetection works.')"


### Optional — snapshot the env so you don't rebuild it every session

The `mm` env does **not** survive a Colab disconnect. Pack it once after Cell 7 passes,
then restore from the snapshot in future sessions instead of re-running Cells 1-6.


In [ ]:
# Save (run once, after Cell 7 passes). Takes a while; ~2-4 GB on Drive.
# conda-pack must be installed in the OUTER/base env (it invokes `conda pack`, not
# `python -m conda_pack`) -- installing it into `mm` via `conda run -n mm pip install`
# (the original bug here) leaves the base `conda` CLI without the `pack` subcommand.
!mkdir -p /content/drive/MyDrive/computer_vision
!pip install -q conda-pack
!conda pack -n mm -o /content/drive/MyDrive/computer_vision/mm_env.tar.gz


In [ ]:
import os

SNAPSHOT = '/content/drive/MyDrive/computer_vision/mm_env.tar.gz'
if not os.path.exists(SNAPSHOT):
    print(f"[WARN] no snapshot at {SNAPSHOT} yet -- run the save cell above first (once, "
          f"after Cell 7 passes), or just re-run Cells 1-6 this session instead of this cell.")
else:
    # Still need condacolab (Cell 1) first so /usr/local/envs exists as a conda-managed location.
    get_ipython().system('mkdir -p /usr/local/envs/mm')
    get_ipython().system(f'tar -xzf {SNAPSHOT} -C /usr/local/envs/mm')
    get_ipython().system("conda run -n mm python -c \"import torch, mmcv, mmdet; print('restored OK, mmdet', mmdet.__version__)\"")


## 2. Task 2 — YOLO -> COCO conversion

Reads `datasets/bdd100k_yolo/{train,val}/images|labels` (confirmed on-Drive layout, see
`scripts/preprocess_data.py`) and writes `datasets/bdd100k_yolo/annotations/{train,val}.json`.

Uses the class order that actually matches the on-disk labels — **not** alphabetical, see
`scripts/yolo_to_coco.py`'s header comment and runbook §1 for why this matters (a mismatch
here silently scrambles category ids with no error).


In [ ]:
!conda run -n mm python scripts/yolo_to_coco.py


In [ ]:
%%bash
# Acceptance check: pycocotools loads both files without error, and annotation count
# roughly matches non-empty label-file line count.
conda run -n mm python << 'PY'
from pycocotools.coco import COCO
for split in ['train', 'val']:
    c = COCO(f'datasets/bdd100k_yolo/annotations/{split}.json')
    print(split, '-> images:', len(c.imgs), '| annotations:', len(c.anns), '| categories:', len(c.cats))
PY


## 3. Task 3 — RTMDet config sanity checks

Before trusting `configs/rtmdet_bdd100k.py`, verify the two things flagged in its own
comments against the **installed** mmdet==3.3.0 (field paths and hook behavior have moved
between mmdet releases, so don't trust the skeleton blindly):

1. `bbox_head.num_classes` field path resolves correctly on the base RTMDet-s config.
2. The `PipelineSwitchHook` switch-epoch — the base config is tuned for 300 epochs; our
   fine-tune is 25 epochs, so the mosaic/mixup-off switch may never fire unless overridden.


In [ ]:
%%bash
conda run -n mm python << 'PY'
from mmengine import Config
c = Config.fromfile('mmdet::rtmdet/rtmdet_s_8xb32-300e_coco.py')
print('base bbox_head.num_classes:', c.model.bbox_head.num_classes)
for hook in c.custom_hooks:
    if 'PipelineSwitch' in hook.get('type', ''):
        print('PipelineSwitchHook switch_epoch:', hook.get('switch_epoch'))
PY


In [ ]:
%%bash
# Loads our actual fine-tune config and confirms num_classes took effect (should be 10).
conda run -n mm python << 'PY'
from mmengine import Config
c = Config.fromfile('configs/rtmdet_bdd100k.py')
print('fine-tune bbox_head.num_classes:', c.model.bbox_head.num_classes)
print('max_epochs:', c.train_cfg.max_epochs)
PY


### 1-image overfit sanity check (cheap, ~1-2 min on T4)

Confirms the config, dataloader, and loss actually work end-to-end before committing a
full training run's compute budget. Loss should visibly decrease over a handful of iters.


In [ ]:
# TODO before running: point train_dataloader at a 1-image subset, e.g. by adding
#   train_dataloader = dict(dataset=dict(indices=1))
# to a throwaway copy of the config, or pass --cfg-options train_dataloader.dataset.indices=1
# on the command line if your mmdet build's train.py accepts --cfg-options (mmdet 3.x does).
!MPLBACKEND=Agg conda run -n mm python -m mmdet.tools.train configs/rtmdet_bdd100k.py \
    --cfg-options train_dataloader.dataset.indices=1 train_cfg.max_epochs=1 train_cfg.val_interval=1


## 4. Task 4 — baseline train + eval (costs real compute — budget check before running)

Reproduces the Phase 3 baseline inside mmdet. Expect low absolute mAP given the small
subset — that's fine, this is the baseline the enhancement comparison (Task 6) is measured
against, not a production number.


In [ ]:
!MPLBACKEND=Agg conda run -n mm python -m mmdet.tools.train configs/rtmdet_bdd100k.py


## 5. Task 6 — with/without enhancement evaluation (the thesis result)

Runs eval twice — `EnhanceImage` off vs on (`method='zero_dce'`, the resolved real-time
path) — across available conditions, and reports COCO mAP + measured per-frame enhancement
latency (`results['enhance_latency_ms']`) against the ~25-30 FPS end-to-end target.

Wire `EnhanceImage` into `test_pipeline` (see `scripts/mm_transforms.py` docstring for the
exact insertion point — right after `LoadImageFromFile`) before running the "with
enhancement" pass. Keep a copy of the config without it for the "without" baseline pass.


In [ ]:
# Baseline (no enhancement) — uses configs/rtmdet_bdd100k.py + the checkpoint from Task 4.
!MPLBACKEND=Agg conda run -n mm python -m mmdet.tools.test configs/rtmdet_bdd100k.py \
    work_dirs/rtmdet_bdd100k/latest.pth --out results_baseline.pkl


In [ ]:
# With enhancement — point at a config variant that adds EnhanceImage to test_pipeline
# (e.g. configs/rtmdet_bdd100k_enhanced.py, once you've created it as a small delta config
# with custom_imports=['scripts.mm_transforms'] and the EnhanceImage insertion).
!MPLBACKEND=Agg conda run -n mm python -m mmdet.tools.test configs/rtmdet_bdd100k_enhanced.py \
    work_dirs/rtmdet_bdd100k/latest.pth --out results_enhanced.pkl
